In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "blue"
    least_favourite_colour: str = "yellow"

In [8]:
from langchain.agents import create_agent

agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    context_schema=ColourContext  
)

In [9]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

In [10]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='f4fcc21b-bb1e-42f5-9334-01372f47d528'),
              AIMessage(content='I’m not sure—what’s your favourite colour?', additional_kwargs={'reasoning_content': 'The user asks: "What is my favourite colour?" This is a question about the user. There\'s no prior context. We cannot know the user\'s favorite color. According to policy, we must not guess or provide misinformation. We can say we don\'t know. We can ask the user to tell us. This is permissible. The user might want to see if we can guess or maybe we can ask. According to policy: "If the user asks for personal preference, we must ask them." So answer: "I don\'t know, what is your favourite color?" We must not guess. So we respond politely asking.'}, response_metadata={'token_usage': {'completion_tokens': 144, 'prompt_tokens': 77, 'total_tokens': 221, 'completion_time': 0.150252991, 'completion_tokens_details': {'reaso

In [11]:
print(response["messages"][-1].content) #  مش هيعرف يجاب مالوش اكسس

I’m not sure—what’s your favourite colour?


## Accessing Context

In [12]:
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime[ColourContext]) -> str:
    """Get the favourite colour of the user"""
    return runtime.context.favourite_colour

@tool
def get_least_favourite_colour(runtime: ToolRuntime[ColourContext]) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

In [14]:
agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext
)

In [15]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='2c8872ab-7acd-40d2-ae62-5ca36c7aa6f6'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks "What is my favourite colour?" We have a tool to get favourite colour. We should call that.', 'tool_calls': [{'id': 'fc_14b4a7a8-6bce-430c-a26f-cdc2473ad20f', 'function': {'arguments': '{}', 'name': 'get_favourite_colour'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 149, 'total_tokens': 194, 'completion_time': 0.049320888, 'completion_tokens_details': {'reasoning_tokens': 24}, 'prompt_time': 0.008334079, 'prompt_tokens_details': None, 'queue_time': 0.153763322, 'total_time': 0.057654967}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_976a85d17a', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a09a81-886f-7

In [16]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(favourite_colour="green")
)

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='bb31ad32-1f0c-4d37-803d-644fcd9f0a72'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call the function get_favourite_colour.', 'tool_calls': [{'id': 'fc_da2d014b-6f30-4507-9e8f-4bc3cb90aeb1', 'function': {'arguments': '{}', 'name': 'get_favourite_colour'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 149, 'total_tokens': 182, 'completion_time': 0.033642552, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.007278908, 'prompt_tokens_details': None, 'queue_time': 0.193349049, 'total_time': 0.04092146}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_3cdd24b83b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a09a81-8b09-7203-99c5-dc0735e473e3-0', tool_calls=[{'name': 'get_

In [17]:
print(response["messages"][-1].content)

Your favourite colour is **green**.
